# Custom ADS SDK Deployment

Authour: Assaf Rabinowicz\
Date: April 2026

### The following notebook covers:
1. Building a model to predict Titanic survival using a Scikit-learn pipeline and validate the model
2. Preparing model artifacts using the ADS SDK
3. Customizing the generated model artifact to incorporate feature engineering into the deployment script
4. Registering, deploying, and invoking the model

### Runtime:
- generalml_p311_cpu_x86_64_v1 
- Additional libraries were installed, including category_encoders and seaborn

## Packages and Data  Import

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import requests

import oci
from oci.auth.signers import get_resource_principals_signer
import ads
from ads.model.generic_model import GenericModel

from pre_processing import create_features, split_column_types, build_preprocessor
from models import build_model, build_pipeline, evaluate_model,optimize_hyperparameters

ads.set_auth(auth='resource_principal')

In [ ]:
raw = fetch_openml(name="titanic", version=1, as_frame=True)
df = raw.frame.copy()

## Data Processing

In [ ]:
df["survived"] = pd.to_numeric(df["survived"], errors="coerce")
df = df.dropna(subset=["survived"])
df["survived"] = df["survived"].astype(int)

df.head()

In [ ]:
df_processed = create_features(df)

target_col = "survived"
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_cols, low_card_cols, high_card_cols = split_column_types(X_train)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Low-card categorical columns: {low_card_cols}")
print(f"High-card categorical columns: {high_card_cols}")

## Modeling

In [ ]:
preprocessor = build_preprocessor(numeric_cols, low_card_cols, high_card_cols) # build_preprocessor uses category_encoders liberary. You can install it, or alternatively use OneHotEncoder for all categorical variales
model = build_model(random_state=42) # using RandomForestClassifier model
pipeline = build_pipeline(preprocessor, model)
best_pipeline = optimize_hyperparameters(pipeline, X_train, y_train) #best model of hyperparameter optimization is selected

## Validation

In [ ]:
results = evaluate_model(best_pipeline, X_test, y_test)

print("Classification report:")
print(results["classification_report"])
print(f"ROC-AUC: {results['roc_auc']:.4f}")

cm = results["confusion_matrix"]
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Preparing Model Artifacts Using ADS

In [ ]:
X_sample = X.iloc[[0]]
artifact_dir = '<your model artifacts directory path>'

In [ ]:
# Here we use GenericModel for creating the model artifacts. 
#Another alternative for sklearn models\pipelines is: ads.model.framework.sklearn_model.SklearnModel
artifact_object = GenericModel(estimator=best_pipeline,
                            artifact_dir=artifact_dir)

### Important Note:
- If you specify inference_conda_env=generalml_p311_cpu_x86_64_v1 in the prepare function below, the base version of this environment will be used, which does not include the category_encoders library.
- Therefore, if your scoring depends on category_encoders, you must publish an updated conda environment that includes this library and provide its URL, as shown in the example below.
- For more details on publishing a conda environment, see: https://docs.oracle.com/en-us/iaas/Content/data-science/using/conda_publishs_object.htm

In [ ]:
artifact_object.prepare(
    inference_conda_env='oci://pub-conda-env@<name_space>/cpu/generalml_p311_cpu_x86_64_v1/1.0/generalml_p311_cpu_x86_64_v1',
    inference_python_version="3.11",
    X_sample=X_sample,
    force_overwrite=True
)

## Customizing the Model Artifacts

1. Add the create_features logic in the score.py:
    1. Copy the create_features function from pre_processing.py and paste it into score.py
    2. Modify the predict function to call create_features after pre_inference:\
    features = pre_inference(data, input_schema_path)\
    features = create_features(features) # added to apply the customization
    yhat = post_inference(model.predict(features))
2. Update the deserialize function in score.py
    1. add 'import StringIO'
    2. Wrap the json_data in the str path:\
    if "pandas.core.frame.DataFrame" in data_type or isinstance(json_data, str):\
    return pd.read_json(StringIO(json_data), dtype=fetch_data_type_from_schema(input_schema_path)) # add StringIO for better practice
3. In case of installing new libareries requires to scoring (like category_encoders in our case):
    1. Publish your custom conda environment
    2. Update the runtime.yaml file by replacnig the INFERENCE_ENV_PATH value with the path to your published custom conda environment. Example:\
        INFERENCE_ENV_PATH: 'oci://pub-conda-env@<'your_name_space'>/conda_environments/cpu/General Machine Learning for CPUs on Python 3.11/1.0/generalml_p311_cpu_x86_64_v1'
4. Another option to modify score.py is to add the score_py_uri argument to the prepare() method rather than adjusting it ad hoc. The score_py_uri argument points to a pre-created score.py file.

In [ ]:
# Reloading in required in order to update the model in the memory for the next steps
artifact_object.reload()

## Testing the Model Locally

Two tests are recommended:
1. Verify method – validates the logic in the score.py file by testing the model’s inference pipeline end-to-end.
2. Introspect – ensures that the model artifact directory is correctly structured and contains all required components.

Both tests must pass before model registration.

In [ ]:
sample = raw.frame.iloc[[0]].copy()

test_input = {
    "data": sample.to_json(orient="records")
}
artifact_object.verify(test_input)

In [ ]:
artifact_object.introspect()

## Saving and Deploying the Model

In [ ]:
model_id = artifact_object.save(
    display_name="titanic-survived-pipeline", # All arguments are optional; if not provided, they will be replaced with default values.
    description="sklearn classification pipeline with feature engineering",
    ignore_pending_changes=True)

print(f"Model saved: {model_id}")

In [ ]:
deployed_model = artifact_object.deploy(
    display_name="titanic-survived-pipeline deployment", # All arguments are optional; if not provided, they will be replaced with default values.
    deployment_log_group_id="<your_log_group_ocid>",
    deployment_predict_log_id="<your_log_ocid>",
    deployment_instance_shape="VM.Standard.E4.Flex",
    deployment_ocpus=1,
    deployment_memory_in_gbs=16
)

## Scoring

In [ ]:
endpoint = deployed_model.url + "/predict" # you can also find the endpoint in the UI Console

signer = get_resource_principals_signer()
response = requests.post(
    endpoint,
    json=test_input,
    auth=signer
)
print(response.json())